In [1]:
import pandas as pd

# Load your Excel file
file_path = "techniques.xlsx"   # change to your actual filename
df = pd.read_excel(file_path)

# Ensure ID column is string
df["ID"] = df["ID"].astype(str)

# Identify techniques and sub-techniques
df["is_subtechnique"] = df["ID"].str.contains(r"\.")

# Count totals
total_techniques = df[~df["is_subtechnique"]]["ID"].nunique()
total_subtechniques = df[df["is_subtechnique"]]["ID"].nunique()

print("Total Techniques:", total_techniques)
print("Total Sub-techniques:", total_subtechniques)
print("Grand Total (Techniques + Sub-techniques):", total_techniques + total_subtechniques)

# Optional: count sub-techniques per technique
df["parent_technique"] = df["ID"].str.split(".").str[0]

subtechnique_counts = (
    df[df["is_subtechnique"]]
    .groupby("parent_technique")["ID"]
    .nunique()
    .reset_index(name="subtechnique_count")
)

print("\nSub-technique count per technique:")
print(subtechnique_counts)


Total Techniques: 203
Total Sub-techniques: 453
Grand Total (Techniques + Sub-techniques): 656

Sub-technique count per technique:
   parent_technique  subtechnique_count
0             T1001                   3
1             T1003                   8
2             T1011                   1
3             T1016                   2
4             T1020                   1
..              ...                 ...
91            T1601                   2
92            T1602                   2
93            T1606                   2
94            T1608                   6
95            T1614                   1

[96 rows x 2 columns]


In [2]:
import pandas as pd

# Load the data
df = pd.read_excel("cwe_data.xlsx")

# Normalize abstraction values (recommended)
df["Abstraction"] = df["Abstraction"].str.strip().str.title()

# Count each abstraction category
abstraction_counts = df["Abstraction"].value_counts()

print(abstraction_counts)


Base        537
Variant     299
Class       112
Pillar       10
Compound      7
Name: Abstraction, dtype: int64


In [3]:
import pandas as pd

# Load the data
df = pd.read_excel("cwe_data.xlsx")

# Normalize abstraction values
df["Abstraction"] = df["Abstraction"].str.strip().str.title()

# Filter Pillar CWEs
pillars = df[df["Abstraction"] == "Pillar"][["CWE_ID", "Name"]]

print(f"Number of Pillars: {len(pillars)}")
print(pillars)


Number of Pillars: 10
     CWE_ID                                               Name
419     284                            Improper Access Control
568     435  Improper Interaction Between Multiple Correctl...
790     664  Improper Control of a Resource Through its Lif...
806     682                              Incorrect Calculation
816     691               Insufficient Control Flow Management
818     693                       Protection Mechanism Failure
822     697                               Incorrect Comparison
825     703  Improper Check or Handling of Exceptional Cond...
829     707                            Improper Neutralization
832     710             Improper Adherence to Coding Standards


In [2]:
import pandas as pd

# ============================================================
# Load CAPEC dataset (Excel only)
# ============================================================
df = pd.read_excel("capec_data.xlsx")

print("\n================ CAPEC DATASET OVERVIEW ================\n")
print(f"Total number of CAPEC attack patterns: {len(df)}")
print("\nColumns in dataset:")
print(df.columns.tolist())

# ============================================================
# Raw abstraction values check (important!)
# ============================================================
print("\n================ RAW ABSTRACTION VALUES ================\n")
print(df["Abstraction"].unique())

# ============================================================
# Dataset completeness analysis
# ============================================================
print("\n================ DATA COMPLETENESS ======================\n")
missing_counts = (df == "N/A").sum().sort_values(ascending=False)
missing_percent = ((df == "N/A").sum() / len(df) * 100).round(1)

completeness_df = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage (%)": missing_percent
})

print(completeness_df)

# ============================================================
# Abstraction level analysis (corrected for CAPEC)
# ============================================================
print("\n================ ABSTRACTION LEVEL DISTRIBUTION =========\n")

# Ensure proper CAPEC categories are used
# Replace "Meta" or incorrect values with correct CAPEC terms if needed
df["Abstraction"] = df["Abstraction"].replace({
    "Meta": "Mechanism"  # adjust if your data has "Meta"
})

# Define standard CAPEC order
abstraction_order = ["Detailed", "Standard", "Mechanism", "Category"]

abstraction_counts = df["Abstraction"].value_counts().reindex(abstraction_order, fill_value=0)
abstraction_percent = (abstraction_counts / len(df) * 100).round(1)

abstraction_df = pd.DataFrame({
    "Count": abstraction_counts,
    "Percentage (%)": abstraction_percent
})

print(abstraction_df)

# ============================================================
# Likelihood of Attack analysis
# ============================================================
print("\n================ LIKELIHOOD OF ATTACK ===================\n")
likelihood_counts = df["Likelihood_Of_Attack"].value_counts()
likelihood_percent = (likelihood_counts / len(df) * 100).round(1)

likelihood_df = pd.DataFrame({
    "Count": likelihood_counts,
    "Percentage (%)": likelihood_percent
})

print(likelihood_df)

# ============================================================
# Typical Severity analysis
# ============================================================
print("\n================ TYPICAL SEVERITY =======================\n")
severity_counts = df["Typical_Severity"].value_counts()
severity_percent = (severity_counts / len(df) * 100).round(1)

severity_df = pd.DataFrame({
    "Count": severity_counts,
    "Percentage (%)": severity_percent
})

print(severity_df)

# ============================================================
# Behavioural richness analysis
# ============================================================
print("\n================ BEHAVIOURAL METADATA COVERAGE ==========\n")
behavioural_fields = [
    "Execution_Flow",
    "Prerequisites",
    "Skills_Required",
    "Resources_Required",
    "Consequences",
    "Mitigations",
    "Example_Instances"
]

coverage_data = []
for field in behavioural_fields:
    available = (df[field] != "N/A").sum()
    coverage_data.append({
        "Field": field,
        "Available_Count": available,
        "Coverage (%)": round(available / len(df) * 100, 1)
    })

behavioural_df = pd.DataFrame(coverage_data)
print(behavioural_df)

# ============================================================
# CWE linkage analysis
# ============================================================
print("\n================ CWE LINKAGE ANALYSIS ===================\n")
capec_with_cwe = (df["Related_Weaknesses"] != "N/A").sum()
capec_without_cwe = len(df) - capec_with_cwe

print(f"CAPEC patterns linked to at least one CWE: {capec_with_cwe}")
print(f"CAPEC patterns without CWE linkage: {capec_without_cwe}")
print(f"CWE linkage coverage (%): {round(capec_with_cwe / len(df) * 100, 1)}")

# ============================================================
# MITRE ATT&CK linkage analysis
# ============================================================
print("\n================ MITRE ATT&CK LINKAGE ===================\n")
capec_with_attack = (df["Mitre_Techniques"] != "N/A").sum()
capec_without_attack = len(df) - capec_with_attack

print(f"CAPEC patterns mapped to ATT&CK techniques: {capec_with_attack}")
print(f"CAPEC patterns without ATT&CK mapping: {capec_without_attack}")
print(f"ATT&CK mapping coverage (%): {round(capec_with_attack / len(df) * 100, 1)}")

# ============================================================
# Summary statistics (for thesis)
# ============================================================
print("\n================ SUMMARY (THESIS-READY) =================\n")
print(f"- Total CAPEC attack patterns collected: {len(df)}")
print(f"- Abstraction levels represented: {df['Abstraction'].nunique()}")
print(f"- CAPEC entries linked to CWE weaknesses: {round(capec_with_cwe / len(df) * 100, 1)}%")
print(f"- CAPEC entries mapped to MITRE ATT&CK: {round(capec_with_attack / len(df) * 100, 1)}%")

print("\nBehavioural field coverage (%):")
for _, row in behavioural_df.iterrows():
    print(f"- {row['Field']}: {row['Coverage (%)']}%")

print("\n================ END OF ANALYSIS =========================\n")



================ CAPEC DATASET OVERVIEW ================

Total number of CAPEC attack patterns: 615

Columns in dataset:
['Unnamed: 0', 'CAPEC_ID', 'Name', 'Abstraction', 'Status', 'Description', 'Extended_Description', 'Likelihood_Of_Attack', 'Typical_Severity', 'Related_Attack_Patterns', 'Execution_Flow', 'Prerequisites', 'Skills_Required', 'Resources_Required', 'Consequences', 'Mitigations', 'Example_Instances', 'Related_Weaknesses', 'Taxonomy_Mappings', 'Mitre_Techniques']

================ RAW ABSTRACTION VALUES ================

['Standard' 'Detailed' 'Meta']

================ DATA COMPLETENESS ======================

                         Missing_Count  Missing_Percentage (%)
Unnamed: 0                           0                     0.0
CAPEC_ID                             0                     0.0
Name                                 0                     0.0
Abstraction                          0                     0.0
Status                               0             

In [4]:
import xml.etree.ElementTree as ET
import pandas as pd

# ============================================================
# Load CAPEC XML
# ============================================================
capec_file = "capec_v3.9.xml"  # your CAPEC XML file
tree = ET.parse(capec_file)
root = tree.getroot()

# Define XML namespace
namespace = {
    "capec": "http://capec.mitre.org/capec-3",
    "xhtml": "http://www.w3.org/1999/xhtml"
}

# List to store extracted attack patterns
data = []

# Iterate through all attack patterns
for attack_pattern in root.findall(".//capec:Attack_Patterns/capec:Attack_Pattern", namespace):
    capec_id = attack_pattern.get("ID", "N/A")
    name = attack_pattern.get("Name", "N/A")
    abstraction = attack_pattern.get("Abstraction", "N/A")  # <-- this preserves Category, Mechanism, Standard, Detailed
    status = attack_pattern.get("Status", "N/A")

    data.append({
        "CAPEC_ID": capec_id,
        "Name": name,
        "Abstraction": abstraction,
        "Status": status
    })

# Convert to DataFrame
df = pd.DataFrame(data)

# ============================================================
# Show all unique abstraction levels
# ============================================================
print("\n================ RAW ABSTRACTION LEVELS =================\n")
print(df["Abstraction"].value_counts())
print("\nUnique abstraction levels:", df["Abstraction"].unique())

# ============================================================
# Save to Excel for further analysis
# ============================================================
df.to_excel("capec_abstraction_levels.xlsx", index=False)
print("\nExcel saved as 'capec_abstraction_levels.xlsx'")




================ RAW ABSTRACTION LEVELS =================

Abstraction
Detailed    341
Standard    197
Meta         77
Name: count, dtype: int64

Unique abstraction levels: ['Standard' 'Detailed' 'Meta']

Excel saved as 'capec_abstraction_levels.xlsx'
